In [ ]:
%pip install ultralytics supervision

In [ ]:
import cv2 as cv
import supervision as sv
from ultralytics import YOLO
import shutil
print("Supervision Version:", sv.__version__)

In [ ]:
model_path ='market_dataset-2/YOLO_Egitim_Sonuclari/weights/best.pt'
video_giris_yolu = 'market_video.mp4'
video_cikti_lokal= 'market_video_cikti.mp4'



model = YOLO(model_path)
tracker = sv.ByteTrack(track_activation_threshold=0.35,
                       lost_track_buffer=120,
                       minimum_matching_threshold=0.85
                       )

cap = cv.VideoCapture(video_giris_yolu)
assert cap.isOpened(), "Hata: Video dosyası açılamadı. Yolu kontrol et"

w,h,fps = (int(cap.get(x)) for x in (cv.CAP_PROP_FRAME_WIDTH, cv.CAP_PROP_FRAME_HEIGHT, cv.CAP_PROP_FPS))

#çizgi tanımlama
# Paspasın üst kenarı boyunca çizgi — kapıyı tam kesiyor
LINE_START = sv.Point(1130, 720)
LINE_END   = sv.Point(1130, 480)

line_zone = sv.LineZone(
    start=LINE_START,
    end=LINE_END,
    triggering_anchors=[sv.Position.BOTTOM_CENTER,
                        sv.Position.CENTER]
)
# Geçici olarak ekle, konsolda bak
print(f"IN: {line_zone.in_count} | OUT: {line_zone.out_count}")

line_annotator = sv.LineZoneAnnotator(thickness = 4)
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()


video_writer = cv.VideoWriter(video_cikti_lokal, cv.VideoWriter_fourcc(*'mp4v'), fps, (w,h))

people_inside = 14
onceki_kisi_sayisi = 14
frame_count = 0

print(f"Video Boyutu: {w}x{h} | FPS: {fps}")
print(f"Çizgi Koordinatları: {LINE_START.x},{LINE_START.y} -> {LINE_END.x},{LINE_END.y}")
print("-"*50)

while cap.isOpened():
  ret,frame = cap.read()
  if not ret:
    break

  frame_count += 1
  anlik_saniye = frame_count / fps

  #yolo tespiti
  results = model(frame, classes = [0], conf=0.35, verbose=False)[0] #class 0 = person

  detections = sv.Detections.from_ultralytics(results)

  detections = detections.with_nms(threshold=0.5)

  #takip et(her kişiye ıd ata)
  detections = tracker.update_with_detections(detections)

  # DEBUG 
  print(f"\nFrame: {frame_count}")
  print("Tracker ID'leri:", detections.tracker_id)
  # 

  # Kişilerin ayak noktaları
  bottom_points = detections.get_anchors_coordinates(
     sv.Position.BOTTOM_CENTER
  )

  # Ayak noktalarını kırmızı nokta olarak çiz
  for pt in bottom_points:
    cv.circle(
        frame,
        (int(pt[0]), int(pt[1])),
        5,
        (0, 0, 255),
        -1
    )
  #çizgi geçişini kontrol et
  crossed_in, crossed_out = line_zone.trigger(detections)

  print(f"Toplam IN: {line_zone.in_count} | Toplam OUT: {line_zone.out_count}")

  #sayıyı güncelle
  people_inside += crossed_in.sum() - crossed_out.sum()
  people_inside = max(0, people_inside)  # negatife düşmesin

  #DEBUG 
  print("İçerideki kişi sayısı:", people_inside)
  #

  #anlık değişim yakalama kısmı

  if people_inside != onceki_kisi_sayisi:
    fark = people_inside - onceki_kisi_sayisi
    hareket_yonu = "GİRİŞ YAPTI" if fark > 0 else "ÇIKIŞ YAPTI"

    print(f"Hareket Algılandı! Saniye: {anlik_saniye:.2f} | Biri {hareket_yonu} | Güncel Sayı: {people_inside}")
    onceki_kisi_sayisi = people_inside

  #etiketleri ve yazıları videoya işleme
  labels = [f"ID: {tid}" for tid in detections.tracker_id]
  frame = box_annotator.annotate(frame, detections)
  frame = label_annotator.annotate(frame, detections, labels)
  frame = line_annotator.annotate(frame, line_zone)

  #güncel sayıyı gösterme
  cv.putText(frame, f"kisi sayisi: {people_inside}",
            (30, 60), cv.FONT_HERSHEY_SIMPLEX,
                1.2, (0, 255, 0), 3, cv.LINE_AA )

  video_writer.write(frame)

cap.release()
video_writer.release()
print("-"*50 + f"\nİşlem Tamamlandı! Kapanış anında içerideki kişi sayısı: {people_inside}")







In [ ]:
cap_debug = cv.VideoCapture(video_giris_yolu)
fps_d = cap_debug.get(cv.CAP_PROP_FPS)
cap_debug.set(cv.CAP_PROP_POS_FRAMES, int(18 * fps_d))
ret, frame = cap_debug.read()
cap_debug.release()

# Çizgiyi daha sola ve aşağıya taşı — kadının tam üstünden geçsin
cv.line(frame, (1000, 350), (1230, 720), (0, 255, 0), 3)
cv.circle(frame, (1000, 350), 8, (255, 0, 0), -1)
cv.circle(frame, (1230, 720), 8, (0, 255, 255), -1)

# Conf'u çok düşür
results = model(frame, classes=[0], conf=0.10, verbose=False)[0]
detections = sv.Detections.from_ultralytics(results)
bottom_points = detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
for pt in bottom_points:
    cv.circle(frame, (int(pt[0]), int(pt[1])), 8, (0, 0, 255), -1)

# Kaç kişi tespit edildi
print(f"Tespit sayısı: {len(bottom_points)}")
for i, pt in enumerate(bottom_points):
    print(f"  Kişi {i}: x={pt[0]:.0f}, y={pt[1]:.0f}")

cv.imwrite("/content/debug_kadin5.png", frame)
print("Kaydedildi!")